# Milestone - Unit 1: the measured baseline

Every later unit of this book claims a faster car.
A claim is only as good as the number it beats,
so the first milestone pins that number: the official starter agent, `agents/unit-01.py`,
raced five times per track with seeded actuator jitter, and measured.

This notebook is the contract the rest of the project builds on.
It asserts, rather than displays, each fact the unit taught:

- **G1** the starter finishes 3 laps of `oval` and `monza-like` in all 5 runs, with 0 collisions;
- **G2** the per-seed times are exactly the pinned ones (times are quantized to the 0.05 s tick),
  and with jitter 0 the `oval` race is the deterministic 223.50 s;
- **G3** sector splits computed from the `t` and `progress` telemetry columns equal the runner's own,
  sum to the race time, and survive a v1 CSV round trip (the path the real-simulator lab takes);
- **G4** the submission passes `check_submission`;
- **G6** a crash restarts the race with the clock running:
  total = clean time + crash time + the 1.0 s hold.

(G5, the lab check, is a CI check on the lab notebook, not a race.)

## 1. The submission and the pinned numbers

The milestone runs from `projects/racekit/milestones/`, so the submission is one directory up.
The pins come from the unit's probe: `KinematicCar(jitter=0.05)`, seeds 0-4.

In [ ]:
from math import sqrt
from pathlib import Path
from tempfile import TemporaryDirectory

import numpy as np
from racekit.cars import KinematicCar
from racekit.check_submission import check_submission
from racekit.measure import repeat_races, spread
from racekit.plots import plot_speed_map
from racekit.runner import CONTROL_DT, RESPAWN_HOLD_TICKS, load_submission, run_race, shim_installed
from racekit.telemetry import race_meta, read_csv, sector_splits, write_csv

AGENT = Path("..") / "agents" / "unit-01.py"
TOL = 1e-9  # race times are sums of 0.05 s ticks, so "exact" means to float rounding

PINNED = {
    "oval": [220.60, 220.75, 220.75, 220.95, 220.60],
    "monza-like": [870.30, 870.50, 870.70, 870.75, 870.45],
}
DETERMINISTIC_OVAL = 223.50
MONZA_SEED_0_SPLITS = [[99.80, 96.40, 96.55], [96.20, 96.30, 96.45], [96.30, 96.65, 95.65]]

## 2. G1 and G2: five jittered races per track

`repeat_races` builds a fresh car per run from `car_factory` and seeds it with the run's seed,
so the same seed always gives the same race.
One run lies; the median and the range (max - min) of five do not.

In [ ]:
runs = {}
for track in ("oval", "monza-like"):
    runs[track] = repeat_races(AGENT, track, n=5, car_factory=lambda: KinematicCar(jitter=0.05))
    result = runs[track]
    # G1: every run finishes all 3 laps, and none crashes
    assert result.all_finished and result.collisions == 0, (track, result)
    assert all(race.laps == 3 for race in result.results)
    # G2: the pinned per-seed times, exactly
    assert max(abs(result.times - np.array(PINNED[track]))) <= TOL, (track, result.times)
    assert abs(result.median - np.median(PINNED[track])) <= TOL
    assert abs(result.spread - spread(PINNED[track])) <= TOL
    print(f"{track:>10}: times {[round(float(time), 2) for time in result.times]}, "
          f"median {result.median:.2f} s, range {result.spread:.2f} s")

With jitter 0 the car does exactly what it is told, and racekit is fully repeatable:
any seed gives 223.50 s on `oval`, with a range of 0.
CARLA is not repeatable, which is why the jittered runs above are the honest baseline.

In [ ]:
deterministic = repeat_races(AGENT, "oval", n=2, car_factory=lambda: KinematicCar(jitter=0.0))
assert deterministic.all_finished and deterministic.collisions == 0
assert max(abs(deterministic.times - DETERMINISTIC_OVAL)) <= TOL
assert deterministic.spread == 0.0
print(f"jitter 0 oval: {deterministic.median:.2f} s, range {deterministic.spread:.2f} s")

## 3. G3: sector splits from telemetry, and through the CSV

`sector_splits` cuts the race's cumulative `progress` into 3 laps x 3 sectors
and times each from the `t` column.
It must agree with the runner's own `result.sector_splits` and sum to the race time,
both on the in-memory telemetry and on the same race written to a v1 CSV and read back,
because that CSV is exactly what the real-simulator lab analyses.

In [ ]:
def check_splits(race, telemetry, meta):
    splits = sector_splits(telemetry, meta["progress_total"], laps=meta["laps"], per_lap=meta["per_lap"])
    assert splits.shape == (3, 3)
    assert abs(splits - race.sector_splits).max() <= TOL
    assert abs(sum(sum(lap) for lap in splits) - race.total_time) <= TOL
    return splits


for track in ("oval", "monza-like"):
    race = runs[track].results[0]  # seed 0
    meta = race_meta(race)
    splits = check_splits(race, race.telemetry, meta)
    with TemporaryDirectory() as scratch:
        telemetry, read_meta = read_csv(write_csv(race.telemetry, Path(scratch) / f"{track}.csv", meta))
    assert read_meta == meta
    check_splits(race, telemetry, read_meta)
    print(track, "seed 0 splits (s):")
    for lap, row in enumerate(splits, start=1):
        print(f"  lap {lap}: {[round(float(split), 2) for split in row]}")

assert abs(splits - np.array(MONZA_SEED_0_SPLITS)).max() <= TOL

The `monza-like` splits are flat: every sector takes about 96 s,
except lap 1 sector 1, about 3.4 s slower, which is the standing start.
Flat splits are the signature of a speed cap: the starter holds 20 m/s everywhere,
so it loses time uniformly, not in particular corners.
The speed map says the same thing.

In [ ]:
monza = runs["monza-like"].results[0]
ax = plot_speed_map(monza.telemetry, "monza-like")
ax.set_title("monza-like, seed 0: the starter holds about 20 m/s everywhere")
speed = monza.telemetry["v"]
offset = monza.telemetry["offset"]
print(f"mean speed {np.mean(speed):.2f} m/s, max |offset| {max(abs(offset)):.2f} m")
assert abs(np.mean(speed) - 19.71) < 0.005 and abs(max(abs(offset)) - 0.61) < 0.005

## 4. G4: the submission check

`check_submission` parses the file as Python 3.8, applies the deny-list and the shape checks,
and races it once.
It is not a proof that the file runs on 3.8: the routed isolated 3.8 test does that,
for every file in `agents/`.

In [ ]:
report = check_submission(AGENT)
print(report)
assert report.ok, report.findings

## 5. G6: what a crash really costs

The pinned race code respawns a crashed car at the **start pose**,
holds it for 20 ticks (1.0 s) without calling the agent,
and resets progress to 0: the race restarts with the clock running.
(The published season report says "a crash costs a lap"; this book follows the pinned code.)

The saboteur is the starter plus a firm right at full throttle from t = 60 s until it crashes.
On `oval` with jitter 0 its total must be 223.50 + t_crash + 1.0, to within one tick.

In [ ]:
with shim_installed():
    Starter, _ = load_submission(AGENT)

STEER_OFF = {"throttle": 1.0, "steer": 0.25, "brake": 0.0, "hand_brake": 0.0, "reverse": 0, "target_gear": 0}


class Saboteur(Starter):
    async def initialize(self):
        await super().initialize()
        self.steps = 0
        self.crashed = False

    async def step(self):
        impulse = self.collision_sensor.get_last_gym_observation()
        if sqrt(sum(float(c) ** 2 for c in impulse)) > 100.0:
            self.crashed = True  # the first step after the hold sees the major-collision impulse
        await super().step()
        if not self.crashed and self.steps * CONTROL_DT >= 60.0:
            await self.vehicle.apply_action(STEER_OFF)
        self.steps += 1


crash = run_race(Saboteur, "oval", car=KinematicCar(jitter=0.0))
assert crash.finished and crash.collisions == 1 and len(crash.respawn_times) == 1
t_crash = crash.respawn_times[0]
assert abs(crash.total_time - (DETERMINISTIC_OVAL + t_crash + 1.0)) <= CONTROL_DT + TOL

held = crash.telemetry["held"]
progress = crash.telemetry["progress"]
first_held = int(np.argmax(held == 1))
assert int(sum(held)) == RESPAWN_HOLD_TICKS == 20
assert progress[first_held - 1] > 100  # well into lap 1 when it crashed ...
assert progress[first_held + RESPAWN_HOLD_TICKS] <= 1  # ... and back to the start after the hold
print(f"crash at {t_crash:.2f} s; total {crash.total_time:.2f} s "
      f"= {DETERMINISTIC_OVAL:.2f} + {t_crash:.2f} + 1.00")
assert abs(t_crash - 60.60) <= TOL and abs(crash.total_time - 285.10) <= TOL

## The baseline, pinned

| | `oval` | `monza-like` |
|---|---|---|
| median of 5 (s) | 220.75 | 870.50 |
| range of 5 (s) | 0.35 | 0.45 |
| collisions | 0 | 0 |

A later unit's agent beats this baseline only when its median is lower by more than the larger of the two ranges
(`noise_aware_better`), and only at 0 collisions:
one crash cost the saboteur 61.6 s, more than a quarter of a clean race.